
# 3. Notebook Modélisation XAI
cells_03 = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# 03 - Modélisation avec XAI\n",
            "\n",
            "Entraînement de modèles avec explicabilité (SHAP, LIME)."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "import pandas as pd\n",
            "import numpy as np\n",
            "import pickle\n",
            "import matplotlib.pyplot as plt\n",
            "import seaborn as sns\n",
            "import sys\n",
            "sys.path.append('../src')\n",
            "\n",
            "from xai_clinical.models.classifiers import XAIClassifier\n",
            "from xai_clinical.explainability.shap_explainer import SHAPExplainer\n",
            "from xai_clinical.explainability.lime_explainer import LIMEExplainer\n",
            "from xai_clinical.evaluation.metrics import compute_classification_metrics\n",
            "\n",
            "from sklearn.ensemble import RandomForestClassifier\n",
            "from sklearn.linear_model import LogisticRegression\n",
            "import warnings\n",
            "warnings.filterwarnings('ignore')"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 1. Chargement des Données"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Charger données prétraitées\n",
            "with open('../data/processed/preprocessed_data.pkl', 'rb') as f:\n",
            "    data = pickle.load(f)\n",
            "\n",
            "X_train = data['X_train']\n",
            "X_test = data['X_test']\n",
            "y_train = data['y_train']\n",
            "y_test = data['y_test']\n",
            "feature_names = data['feature_names']\n",
            "\n",
            "print(f\"Train: {X_train.shape}, Test: {X_test.shape}\")"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 2. Entraînement des Modèles"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Modèle 1: Random Forest\n",
            "rf_model = RandomForestClassifier(\n",
            "    n_estimators=100,\n",
            "    max_depth=10,\n",
            "    random_state=42,\n",
            "    n_jobs=-1\n",
            ")\n",
            "rf_model.fit(X_train, y_train)\n",
            "\n",
            "# Modèle 2: Logistic Regression\n",
            "lr_model = LogisticRegression(max_iter=1000, random_state=42)\n",
            "lr_model.fit(X_train, y_train)\n",
            "\n",
            "print('Modèles entraînés')"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Évaluation\n",
            "models = {'Random Forest': rf_model, 'Logistic Regression': lr_model}\n",
            "\n",
            "for name, model in models.items():\n",
            "    y_pred = model.predict(X_test)\n",
            "    y_prob = model.predict_proba(X_test)\n",
            "    \n",
            "    metrics = compute_classification_metrics(y_test, y_pred, y_prob)\n",
            "    \n",
            "    print(f\"\\n=== {name} ===\")\n",
            "    print(f\"Accuracy: {metrics['accuracy']:.3f}\")\n",
            "    print(f\"F1-Score: {metrics['f1']:.3f}\")\n",
            "    print(f\"AUC-ROC: {metrics.get('auc_roc', 'N/A')}\")"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 3. Explicabilité avec SHAP"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Initialiser SHAP explainer\n",
            "shap_explainer = SHAPExplainer(rf_model, X_train, feature_names)\n",
            "\n",
            "# Expliquer prédictions test\n",
            "shap_values = shap_explainer.explain(X_test[:100])  # subset pour rapidité\n",
            "\n",
            "# Visualisation summary plot\n",
            "shap_explainer.plot_summary(shap_values, max_display=10)"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Explication individuelle\n",
            "instance_idx = 0\n",
            "shap_explainer.plot_waterfall(shap_values, instance_idx)"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 4. Explicabilité avec LIME"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# LIME explainer\n",
            "lime_explainer = LIMEExplainer(\n",
            "    rf_model,\n",
            "    X_train,\n",
            "    feature_names,\n",
            "    mode='classification'\n",
            ")\n",
            "\n",
            "# Expliquer une instance\n",
            "exp = lime_explainer.explain_instance(X_test[0], num_features=10)\n",
            "lime_explainer.plot_explanation(exp)"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 5. Comparaison des Explications"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Comparer features importantes SHAP vs LIME\n",
            "shap_importance = shap_explainer.get_feature_importance()\n",
            "lime_importance = lime_explainer.get_feature_importance(X_test[0])\n",
            "\n",
            "comparison = pd.DataFrame({\n",
            "    'SHAP': shap_importance.head(10).values,\n",
            "    'LIME': lime_importance.head(10).values\n",
            "}, index=shap_importance.head(10).index)\n",
            "\n",
            "comparison.plot(kind='bar', figsize=(12, 6))\n",
            "plt.title('Comparaison Importance Features: SHAP vs LIME')\n",
            "plt.tight_layout()\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 6. Sauvegarde"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Sauvegarder modèles et explainer\n",
            "import joblib\n",
            "\n",
            "joblib.dump(rf_model, '../models/saved_models/random_forest.pkl')\n",
            "joblib.dump(lr_model, '../models/saved_models/logistic_regression.pkl')\n",
            "\n",
            "# Sauvegarder explainers\n",
            "with open('../models/saved_models/shap_explainer.pkl', 'wb') as f:\n",
            "    pickle.dump(shap_explainer, f)\n",
            "\n",
            "print('Modèles et explainers sauvegardés')"
        ]
    }
]

notebook_03 = notebook_structure.copy()
notebook_03["cells"] = cells_03

with open(f"{base_path}/notebooks/03_modelisation_xai.ipynb", "w") as f:
    json.dump(notebook_03, f, indent=1)

print("✅ 03_modelisation_xai.ipynb créé")
